<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/module34a/Lab08.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Lab 8 — VQE for a Materials Model: the Transverse-Field Ising Chain

**Maps to:** Module 4, applied outside chemistry

**Time:** ~60 minutes (instructor walkthrough ~12 min). Total compute ~2 minutes.

---

### Why leave chemistry

Nothing in VQE is chemical. The algorithm needs exactly two things: a Hamiltonian written
as a sum of Pauli strings, and a parameterized circuit. Swap the Hamiltonian and the same
machinery answers questions about **materials** — magnetic ordering, ferroelectric
switching, the response of a coating to an applied field.

The transverse-field Ising model (TFIM) is the standard first stop:

$$ H = -J\sum_{i} Z_i Z_{i+1} \;-\; h\sum_i X_i .$$

* $-J\,Z_iZ_{i+1}$: neighbouring spins want to **align** — this is what makes a magnet.
* $-h\,X_i$: an applied transverse field wants each spin to point sideways instead, i.e.
  into a superposition of up and down.

Two terms that want incompatible things. Turn up $h/J$ and the material undergoes a
**quantum phase transition**: ordered (magnetic) below $h/J\approx1$, disordered above.
That competition — not any single number — is what materials modelling is usually after.

### After this lab you can
1. Build a lattice-model Hamiltonian as a `SparsePauliOp` for any chain length.
2. Use a **hardware-efficient** ansatz where no chemistry-derived ansatz exists.
3. Locate a phase transition by computing order parameters from VQE states.
4. Diagnose the two things that actually go wrong in practice: insufficient ansatz
   expressibility, and local minima.

In [ ]:
# %pip install -q qiskit qiskit-aer matplotlib scipy
import numpy as np, time
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from qiskit import transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit.library import efficient_su2
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2

np.set_printoptions(precision=4, suppress=True)
sim = AerSimulator()
estimator = EstimatorV2(options={"default_precision": 0.0})
print("ready")

## 1. Building the Hamiltonian

A Pauli string on $n$ qubits is just a label like `"IZZI"`. Remember Qiskit prints
$q_{n-1}\dots q_0$, so we build the list of characters in qubit order and reverse it at
the end.

### Exercise 1 — write the model

In [ ]:
def pauli_label(n, ops):
    '''Build a Qiskit Pauli label from {qubit_index: 'X'/'Y'/'Z'} on n qubits.'''
    s = ["I"] * n
    for q, p in ops.items():
        s[q] = p
    return "".join(s[::-1])

def tfim(n, J=1.0, h=1.0, periodic=False):
    '''Transverse-field Ising chain: -J sum ZZ  -  h sum X.'''
    terms = []
    bonds = [(i, i+1) for i in range(n-1)] + ([(n-1, 0)] if periodic else [])
    for i, j in bonds:
        # TODO: append a ZZ term on qubits i and j with coefficient -J
        ...
    for i in range(n):
        # TODO: append an X term on qubit i with coefficient -h
        ...
    return SparsePauliOp.from_list(terms)

N = 4
H = tfim(N, J=1.0, h=0.5)
print(H)
assert len(H) == (N - 1) + N
print(f"\nPASS: {N-1} bond terms + {N} field terms = {len(H)} Pauli strings")

### Exercise 2 — sanity-check the two limits

Before trusting any optimizer, check the cases you can do in your head.

* $h = 0$: the ground state is all spins aligned. Energy $= -J(n-1)$ for an open chain.
* $J = 0$: every spin points along $+x$. Energy $= -h\,n$.

In [ ]:
def exact_ground(H):
    w, v = np.linalg.eigh(H.to_matrix())
    return w[0], v[:, 0]

e_ferro, _ = exact_ground(tfim(N, J=1.0, h=0.0))
e_para,  _ = exact_ground(tfim(N, J=0.0, h=1.0))
print(f"h = 0 : E = {e_ferro:.4f}   expected {-1.0*(N-1):.4f}")
print(f"J = 0 : E = {e_para:.4f}   expected {-1.0*N:.4f}")
assert np.isclose(e_ferro, -(N-1)) and np.isclose(e_para, -N)
print("\nPASS")

## 2. A hardware-efficient ansatz

For H$_2$ we derived the ansatz from chemistry (UCCSD). Here there is no equivalent
theory handing us the right excitations, so we use the other strategy from your
Module 4 slides: a **heuristic, hardware-efficient** circuit. Layers of single-qubit
rotations, alternating with a fixed pattern of entangling gates that matches the device's
connectivity.

`efficient_su2` is Qiskit's standard version: $R_y$ and $R_z$ on every qubit, then a
linear chain of CNOTs, repeated `reps` times.

**The trade-off is explicit.** More `reps` = more expressive = deeper = noisier, and more
parameters for the optimizer to get lost in.

In [ ]:
def make_ansatz(n, reps=2):
    circ = efficient_su2(n, reps=reps, entanglement="linear")
    return transpile(circ, sim, optimization_level=1)   # Aer needs concrete gates

ansatz = make_ansatz(N, reps=2)
print("qubits    :", ansatz.num_qubits)
print("parameters:", ansatz.num_parameters)
print("depth     :", ansatz.depth(), " CNOTs:", ansatz.count_ops().get("cx", 0))
print()
print(efficient_su2(N, reps=1, entanglement="linear").decompose().draw(output="text", fold=100))

## 3. The VQE loop, with restarts

One new ingredient compared with Lab 5: **restarts**. A one-parameter landscape has one
valley. A 24-parameter landscape has many, and COBYLA will happily settle into a bad one.
The standard defence is to run from several random starting points and keep the best.

In [ ]:
def vqe(H, ansatz, n_restarts=3, maxiter=500, seed0=0):
    best_e, best_x = np.inf, None
    def f(x):
        return float(estimator.run([(ansatz, H, x)]).result()[0].data.evs)
    for s in range(n_restarts):
        rng = np.random.default_rng(seed0 + s)
        x0 = rng.uniform(-np.pi/4, np.pi/4, ansatz.num_parameters)
        res = minimize(f, x0, method="COBYLA", options={"maxiter": maxiter})
        if res.fun < best_e:
            best_e, best_x = res.fun, res.x
    return best_e, best_x

t0 = time.time()
H = tfim(N, J=1.0, h=1.0)
e_vqe, x_opt = vqe(H, ansatz)
e_ex, _ = exact_ground(H)
print(f"VQE   : {e_vqe:.6f}")
print(f"exact : {e_ex:.6f}")
print(f"error : {e_vqe - e_ex:.6f}   ({time.time()-t0:.1f} s)")
assert e_vqe - e_ex > -1e-8, "below the true ground state means a bug"

## 4. Sweeping the field: finding the phase transition

Now the physics. Run VQE across a range of $h/J$ and, at each optimum, measure two
**order parameters**:

* **Transverse magnetization** $\langle X\rangle = \frac1n\sum_i \langle X_i\rangle$ —
  how strongly the spins follow the applied field.
* **Nearest-neighbour correlation** $\frac{1}{n-1}\sum_i\langle Z_iZ_{i+1}\rangle$ —
  how strongly the material remains magnetically ordered.

These are what an experimentalist would actually measure on a sample.

Runtime: ~90 seconds.

In [ ]:
def observable_mean(ansatz, x, terms):
    '''Average expectation value of a list of Pauli labels at parameters x.'''
    op = SparsePauliOp.from_list([(t, 1.0/len(terms)) for t in terms])
    return float(estimator.run([(ansatz, op, x)]).result()[0].data.evs)

X_terms  = [pauli_label(N, {i: "X"}) for i in range(N)]
ZZ_terms = [pauli_label(N, {i: "Z", i+1: "Z"}) for i in range(N-1)]

t0 = time.time()
h_values = np.round(np.linspace(0.0, 2.5, 9), 3)
E_vqe, E_exact, mag_x, corr_zz = [], [], [], []

for h in h_values:
    Hh = tfim(N, J=1.0, h=h)
    e, x = vqe(Hh, ansatz, n_restarts=3, maxiter=500)
    E_vqe.append(e)
    E_exact.append(exact_ground(Hh)[0])
    mag_x.append(observable_mean(ansatz, x, X_terms))
    corr_zz.append(observable_mean(ansatz, x, ZZ_terms))
    print(f"h={h:5.3f}  E_vqe={e:9.5f}  E_exact={E_exact[-1]:9.5f}  "
          f"err={e-E_exact[-1]:7.4f}  <X>={mag_x[-1]:+.3f}  <ZZ>={corr_zz[-1]:+.3f}")

print(f"\ntotal {time.time()-t0:.1f} s")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(h_values, E_exact, "k-", lw=2, label="exact")
ax[0].plot(h_values, E_vqe, "o", ms=6, label="VQE")
ax[0].set_xlabel("h / J"); ax[0].set_ylabel("ground state energy")
ax[0].set_title(f"TFIM, N = {N}"); ax[0].legend(fontsize=8)

ax[1].plot(h_values, corr_zz, "s-", label=r"$\langle Z_iZ_{i+1}\rangle$  (magnetic order)")
ax[1].plot(h_values, mag_x, "o-", label=r"$\langle X_i\rangle$  (follows the field)")
ax[1].axvline(1.0, color="r", ls=":", label="critical point $h/J = 1$")
ax[1].set_xlabel("h / J"); ax[1].set_ylabel("order parameter")
ax[1].set_title("Ordered $\\rightarrow$ disordered"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

### Exercise 3 — read the transition

Answer below:

* At which $h/J$ do the two curves cross? The infinite chain has its critical point at
  exactly $h/J = 1$; a 4-site chain smears it out. Which direction does the finite size
  push your crossing, and would you expect $N=8$ to be sharper?
* Where along the sweep is the VQE error largest? Explain why the **critical point** is
  the hardest place for a fixed-depth ansatz. (Hint: what happens to the range of
  entanglement in the ground state there?)

In [ ]:
errors = np.array(E_vqe) - np.array(E_exact)
print(f"{'h/J':>6}{'VQE error':>12}")
for h, e in zip(h_values, errors):
    bar = "#" * int(200*e)
    print(f"{h:>6.2f}{e:>12.5f}  {bar}")
print(f"\nworst at h/J = {h_values[int(np.argmax(errors))]:.2f}")

## 5. What controls the accuracy

Two knobs, both important, both with a cost.

### Exercise 4 — ansatz depth

In [ ]:
h_test = 1.0
H_test = tfim(N, J=1.0, h=h_test)
e_ref = exact_ground(H_test)[0]

print(f"{'reps':>5}{'params':>8}{'CNOTs':>7}{'depth':>7}{'error':>11}")
for reps in [1, 2, 3]:
    # TODO: build the ansatz at this depth, run vqe, print params/CNOTs/depth/error
    ...

### Exercise 5 — how many restarts do you actually need?

Run 8 independent single-shot optimizations and look at the spread of final energies.

In [ ]:
finals = []
for s in range(8):
    e, _ = vqe(H_test, ansatz, n_restarts=1, maxiter=500, seed0=100 + s)
    finals.append(e - e_ref)
finals = np.array(finals)

print("errors from 8 random starts:", np.round(finals, 4))
print(f"best {finals.min():.5f}   median {np.median(finals):.5f}   worst {finals.max():.5f}")
print(f"\nfactor between best and worst: {finals.max()/max(finals.min(),1e-9):.1f}x")
print("A single run is a lottery ticket. Restarts are not optional for")
print("heuristic ansatze -- and every restart multiplies your QPU bill.")

## 6. Optional — run one point on hardware

If you completed Lab 7, the same pipeline applies unchanged: transpile the ansatz at its
optimal parameters, `apply_layout` the observable, submit. The circuit here is deeper
than the H$_2$ one, so expect a larger bias.

In [ ]:
#SKIP-VERIFY
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as RuntimeEstimator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=N)
pm = generate_preset_pass_manager(optimization_level=2, backend=backend)

bound = ansatz.assign_parameters(x_opt)
isa = pm.run(bound)
isa_H = tfim(N, 1.0, 1.0).apply_layout(isa.layout)

est = RuntimeEstimator(mode=backend)
est.options.resilience_level = 1
est.options.default_shots = 4000
est.options.dynamical_decoupling.enable = True

e_hw = float(est.run([(isa, isa_H)]).result()[0].data.evs)
print(f"hardware : {e_hw:.5f}")
print(f"simulator: {e_vqe:.5f}")
print(f"exact    : {e_ex:.5f}")
print(f"ISA depth {isa.depth()}, 2Q gates "
      f"{sum(v for k,v in isa.count_ops().items() if k in ('cz','cx','ecr'))}")

## Checkpoint

1. Write the Pauli string for a $Z_2Z_3$ term on a 5-qubit chain, in Qiskit's label order.
2. Why is a hardware-efficient ansatz the right choice here but the *wrong* choice for a
   molecule where UCCSD is available? Give one argument each way.
3. The exact energy is always $\le$ the VQE energy. What does it mean if your VQE result
   comes out lower?
4. Your VQE error peaks at $h/J \approx 1$. Name the physical property of the ground state
   that makes that point hard.
5. You want to study $N = 20$ sites. What breaks first — the number of qubits, the number
   of Pauli terms, the classical optimization, or the exact reference calculation?

### What is next
**Lab 9** does correlated *electrons* in a metal — the Hubbard dimer — and closes the
loop by showing that it is the same problem as H$_2$, wearing different clothes.